##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [ ]:
!pip install -q pinecone

From 'pinecone' library import Pinecone, ServerlessSpec.

In [2]:
from pinecone import Pinecone, ServerlessSpec

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [4]:
import time
import os
from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "quickstart"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)
texts = [
    "Vilnius Old Town has a lot of churches and narrow streets.",
    "Mažeikiai has an oil refinery that is very secure.",
    "Basketball is the most popular sport in Lithuania.",
    "Cepelinai are large dumplings made out of potato filled with meat.",
    "Šaltibarščiai is a cold pink beetroot soup.",
]
embeddings = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=texts,
    parameters={"input_type": "passage"},
)
index.upsert(vectors=[
    {"id": f"vec{i}", "values": e["values"], "metadata": {"text": t}}
    for i, (t, e) in enumerate(zip(texts, embeddings))
])
time.sleep(10)

query = "What should I cook for dinner tonight?"
query_embedding = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[query],
    parameters={"input_type": "query"},
)
results = index.query(
    vector=query_embedding[0]["values"],
    top_k=3,
    include_metadata=True,
)

Print search results.

In [5]:
print(results)

QueryResponse(matches=[ScoredVector(id='vec3', score=0.801022828, values=[], metadata={'text': 'Cepelinai are large dumplings made out of potato filled with meat.'}), ScoredVector(id='vec4', score=0.79390806, values=[], metadata={'text': 'Šaltibarščiai is a cold pink beetroot soup.'}), ScoredVector(id='vec1', score=0.710735, values=[], metadata={'text': 'Mažeikiai has an oil refinery that is very secure.'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 13 Nov 2026 13:08:17 GMT', 'content-type': 'application/json', 'content-length': '449', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '169', 'x-envoy-upstream-service-time': '44', 'x-pinecone-response-duration-ms': '170', 'grpc-status': '0', 'server': 'envoy'}))
